In [1]:
# =============================================================================
# NB9-ENHANCED — FedAdapt-EF: DenseNet-121 with Full Paper Diagnostics
#
# This notebook:
#   1. Trains FedAdapt-EF with DenseNet-121 (2 seeds)
#   2. Saves layer-wise sensitivity data per round (NEW)
#   3. Generates ALL paper figures after training
#   4. Saves complete results pkl for NB8/NB10
#
# Fixed model: densenet121
# Target runtime: ~3 hours on Kaggle T4x2
# =============================================================================

MODEL_NAME = "densenet121"

# =============================================================================
# SECTION 0 — INSTALL & IMPORTS
# =============================================================================

import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    import timm
    print(f"timm {timm.__version__} already installed")
except ImportError:
    print("Installing timm...")
    pip_install("timm>=0.9.0")
    import timm

import os, gc, time, pickle, json, warnings, random, copy
from PIL import Image
import numpy as np
import pandas as pd
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
from collections import defaultdict, OrderedDict

warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F_torch
from torch.utils.data import Dataset, DataLoader

try:
    from torch.amp import autocast, GradScaler
    _AMP_NEW = True
except ImportError:
    from torch.cuda.amp import autocast, GradScaler
    _AMP_NEW = False

import torchvision.transforms as transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score,
                             precision_score, recall_score, confusion_matrix,
                             roc_curve)
from tqdm.auto import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap

print("=" * 60)
print("NB9-ENHANCED: FedAdapt-EF DenseNet-121 + Paper Diagnostics")
print("=" * 60)
print(f"PyTorch: {torch.__version__} | timm: {timm.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")


# =============================================================================
# SECTION 1 — MODEL REGISTRY (DenseNet-121 only)
# =============================================================================

MODEL_REGISTRY = {
    "densenet121": {
        "timm_name":    "densenet121",
        "client_batch": 24,
        "backbone_lr":  3e-5,
        "head_lr":      3e-4,
        "description":  "DenseNet-121 (8M) -- Medical imaging standard",
        "color":        "#2ECC71",
    },
}

print(f"\nModel: {MODEL_NAME}")
print(f"  {MODEL_REGISTRY[MODEL_NAME]['description']}")
print(f"  client_batch={MODEL_REGISTRY[MODEL_NAME]['client_batch']}")


# =============================================================================
# SECTION 2 — CONFIGURATION
# =============================================================================

@dataclass
class Config:
    # Paths
    UNIFIED_CSV: str = "/kaggle/input/datasets/ariroyjit/unified-dr-dataset/unified_dataset.csv"
    STATS_JSON:  str = "/kaggle/input/datasets/ariroyjit/unified-dr-dataset/dataset_stats.json"
    SAVE_DIR:    str = "/kaggle/working/results/multimodel"

    # Model
    MODEL_NAME:  str   = MODEL_NAME
    PRETRAINED:  bool  = True
    DROPOUT:     float = 0.4

    # Data splits (must match NB10 centralized)
    TRAIN_RATIO: float = 0.70
    VAL_RATIO:   float = 0.15
    TEST_RATIO:  float = 0.15

    # Training
    BATCH_SIZE:   int   = 128
    FL_ROUNDS:    int   = 25
    LOCAL_EPOCHS: int   = 1

    # LRs from registry
    BACKBONE_LR:  float = MODEL_REGISTRY[MODEL_NAME]["backbone_lr"]
    HEAD_LR:      float = MODEL_REGISTRY[MODEL_NAME]["head_lr"]
    WEIGHT_DECAY: float = 1e-4

    # Backbone freeze
    FL_FREEZE_ROUNDS: int  = 3
    FREEZE_BN_IN_FL:  bool = True

    # Regularization
    GRAD_CLIP:   float = 1.0
    POS_WEIGHT:  float = 2.0

    # FL setup
    NUM_CLIENTS:   int   = 5
    NON_IID_ALPHA: float = 0.5

    # FedAdapt-EF
    FEDPROX_MU:         float = 0.01
    USE_ERROR_FEEDBACK: bool  = True
    EF_ON_CPU:          bool  = True
    USE_ADAPTIVE:       bool  = True
    ADAPTIVE_MIN_RATIO: float = 0.1
    ADAPTIVE_MAX_RATIO: float = 0.5

    # Checkpointing
    SAVE_CHECKPOINTS: bool = True
    CHECKPOINT_EVERY: int  = 5

    # Seeds
    SEEDS: List[int] = field(default_factory=lambda: [2026, 3407])

    # Hardware
    DEVICE:      str  = "cuda" if torch.cuda.is_available() else "cpu"
    NUM_WORKERS: int  = 2
    PIN_MEMORY:  bool = True
    USE_AMP:     bool = True

config = Config()

# Auto-find CSV
for _cp in [
    config.UNIFIED_CSV,
    "/kaggle/input/unified-dr-dataset/unified_dataset.csv",
    "/kaggle/input/datasets/ariroyjit/unified-dr-dataset/unified_dataset.csv",
]:
    if os.path.exists(_cp):
        config.UNIFIED_CSV = _cp
        break

MODEL_SAVE_DIR = Path(config.SAVE_DIR) / MODEL_NAME
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Paper figures directory
PAPER_DIR = Path("/kaggle/working/results/paper_figures")
PAPER_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nConfig ready")
print(f"  Seeds:      {config.SEEDS}")
print(f"  FL rounds:  {config.FL_ROUNDS}")
print(f"  Save dir:   {MODEL_SAVE_DIR}")
print(f"  Paper figs: {PAPER_DIR}")


# =============================================================================
# SECTION 3 — UTILITIES
# =============================================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def worker_init_fn(worker_id):
    np.random.seed(torch.initial_seed() % 2**32 + worker_id)

def interpolate_history(values, eval_mask):
    result = list(values)
    last = None
    for i in range(len(result)):
        if eval_mask[i]:
            last = result[i]
        elif last is not None:
            result[i] = last
    return result


# =============================================================================
# SECTION 4 — DATASET
# =============================================================================

def get_transforms(split='train'):
    try:
        with open(config.STATS_JSON) as f:
            st = json.load(f)
        mean, std = st['mean'], st['std']
    except Exception:
        mean = [0.485, 0.456, 0.406]
        std  = [0.229, 0.224, 0.225]

    if split == 'train':
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])


class DRDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = Image.open(row['image_path']).convert('RGB')
        except FileNotFoundError:
            img = Image.new('RGB', (224, 224), (0, 0, 0))
        if self.transform:
            img = self.transform(img)
        label = torch.tensor(float(row['binary_label']), dtype=torch.float32)
        return img, label


def remap_image_paths(df):
    sample = df['image_path'].iloc[:20].tolist()
    n_valid = sum(1 for p in sample if os.path.exists(p))
    if n_valid >= 18:
        print(f"  Paths OK ({n_valid}/20 exist)")
        return df

    print(f"  Broken paths ({n_valid}/20 exist) -- scanning /kaggle/input/ ...")
    filename_to_real = {}
    total = 0
    for slug in sorted(os.listdir('/kaggle/input')):
        slug_dir = f'/kaggle/input/{slug}'
        count = 0
        for root, _, files in os.walk(slug_dir):
            for f in files:
                if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                    if f not in filename_to_real:
                        filename_to_real[f] = os.path.join(root, f)
                    count += 1
                    total += 1
        if count > 0:
            print(f"    [{slug}]: {count:,} images")

    print(f"  Total indexed: {total:,} images")
    df = df.copy()
    df['image_path'] = df['image_path'].apply(
        lambda p: filename_to_real.get(os.path.basename(p), p)
                  if not os.path.exists(p) else p
    )
    n_fixed = sum(1 for p in df['image_path'].iloc[:20] if os.path.exists(p))
    print(f"  After remap: {n_fixed}/20 valid")
    return df


def load_and_split_data():
    df = pd.read_csv(config.UNIFIED_CSV)
    print(f"\nDataset: {len(df)} images | DR+: {df['binary_label'].sum():.0f} "
          f"({df['binary_label'].mean()*100:.1f}%)")
    df = remap_image_paths(df)

    train_df, temp_df = train_test_split(
        df, test_size=(1 - config.TRAIN_RATIO),
        stratify=df['binary_label'], random_state=42
    )
    val_ratio_adj = config.VAL_RATIO / (config.VAL_RATIO + config.TEST_RATIO)
    val_df, test_df = train_test_split(
        temp_df, test_size=(1 - val_ratio_adj),
        stratify=temp_df['binary_label'], random_state=42
    )
    print(f"  Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
    return train_df, val_df, test_df


def make_non_iid_splits(train_df, num_clients, alpha, seed=42):
    rng = np.random.default_rng(seed)
    labels = train_df['binary_label'].values
    num_classes = len(np.unique(labels))
    client_indices = [[] for _ in range(num_clients)]
    for c in range(num_classes):
        class_idx = np.where(labels == c)[0]
        rng.shuffle(class_idx)
        proportions = rng.dirichlet(alpha * np.ones(num_clients))
        proportions = (proportions * len(class_idx)).astype(int)
        proportions[-1] = len(class_idx) - proportions[:-1].sum()
        start = 0
        for cid, count in enumerate(proportions):
            client_indices[cid].extend(class_idx[start:start+count].tolist())
            start += count

    client_datasets = {}
    for cid in range(num_clients):
        cdf = train_df.iloc[client_indices[cid]].copy()
        if len(cdf) > 20:
            cdf_train, cdf_val = train_test_split(
                cdf, test_size=0.1,
                stratify=cdf['binary_label'].clip(0, 1),
                random_state=seed + cid
            )
        else:
            cdf_train, cdf_val = cdf, cdf.iloc[:0]
        client_datasets[cid] = (
            DRDataset(cdf_train, get_transforms('train')),
            DRDataset(cdf_val,   get_transforms('val')),
        )
    return client_datasets


# =============================================================================
# SECTION 5 — MODEL
# =============================================================================

class DRClassifier(nn.Module):
    def __init__(self, model_name=MODEL_NAME, pretrained=True, dropout=0.4):
        super().__init__()
        self.model_name = model_name
        timm_name = MODEL_REGISTRY[model_name]["timm_name"]

        self.backbone = timm.create_model(
            timm_name, pretrained=pretrained,
            num_classes=0, global_pool='avg',
        )

        self.backbone.eval()
        with torch.no_grad():
            _dummy = torch.zeros(1, 3, 224, 224)
            feature_dim = int(self.backbone(_dummy).shape[-1])
        del _dummy
        print(f"  {model_name}: feature_dim={feature_dim}")

        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, 1),
        )

    def forward(self, x):
        return self.classifier(self.backbone(x)).squeeze(-1)

    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = True

    def freeze_bn_for_fl(self):
        for m in self.modules():
            if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
                m.eval()
                for p in m.parameters():
                    p.requires_grad = False

    def get_param_groups(self, bb_lr, head_lr):
        return [
            {'params': self.backbone.parameters(),   'lr': bb_lr},
            {'params': self.classifier.parameters(),  'lr': head_lr},
        ]


def create_model():
    m = DRClassifier(MODEL_NAME, pretrained=config.PRETRAINED, dropout=config.DROPOUT)
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"  Created {MODEL_NAME} | trainable params: {n:,}")
    return m

def create_client_model(device):
    m = DRClassifier(MODEL_NAME, pretrained=False, dropout=config.DROPOUT)
    return m.to(device)


# =============================================================================
# SECTION 6 — EVALUATION
# =============================================================================

def evaluate(model, loader, device, return_arrays=True):
    m = model.module if isinstance(model, nn.DataParallel) else model
    m.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            probs = torch.sigmoid(m(x)).cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(y.numpy())

    probs  = np.array(all_probs)
    labels = np.array(all_labels)
    preds  = (probs >= 0.5).astype(int)

    metrics = dict(
        auc      = roc_auc_score(labels, probs),
        f1       = f1_score(labels, preds, zero_division=0),
        precision= precision_score(labels, preds, zero_division=0),
        recall   = recall_score(labels, preds, zero_division=0),
        accuracy = accuracy_score(labels, preds),
    )
    if return_arrays:
        fpr, tpr, _ = roc_curve(labels, probs)
        metrics.update(dict(
            cm=confusion_matrix(labels, preds),
            fpr=fpr, tpr=tpr,
            probs=probs, labels=labels, preds=preds,
        ))
    return metrics


# =============================================================================
# SECTION 7 — ADAPTIVE COMPRESSION + ERROR FEEDBACK
# (Enhanced: saves per-layer sensitivity data for paper figures)
# =============================================================================

class SparsePayload:
    def __init__(self, indices, values, shape):
        self.indices = indices
        self.values  = values
        self.shape   = shape

    def to_dense(self, device):
        out = torch.zeros(self.shape, dtype=torch.float32, device=device)
        idx = (torch.from_numpy(self.indices).long()
               if isinstance(self.indices, np.ndarray) else self.indices.long())
        val = (torch.from_numpy(self.values).float()
               if isinstance(self.values,  np.ndarray) else self.values.float())
        out.view(-1)[idx.to(device)] = val.to(device)
        return out.view(self.shape)

    @property
    def nbytes(self):
        return self.indices.nbytes + self.values.nbytes


def adaptive_compress_with_ef(deltas, error_buf):
    """
    Adaptive TopK compression with delta-domain error feedback.
    Returns extra sensitivity diagnostics for paper figures.
    """
    layer_sensitivities = {}
    for name, delta in deltas.items():
        if delta.dtype not in [torch.float32, torch.float16, torch.bfloat16]:
            continue
        d = delta.float()
        n = d.numel()
        if n < 10:
            continue
        layer_sensitivities[name] = d.norm().item() / (n ** 0.5)

    if not layer_sensitivities:
        return deltas, error_buf, 1, 1, {}, 0.0, {}

    sens_values = sorted(layer_sensitivities.values())
    threshold   = sens_values[len(sens_values) // 2]

    compressed     = {}
    new_ef         = {}
    total_orig     = 0
    total_comp     = 0
    layer_ratios   = {}
    ef_norms       = []

    for name, delta in deltas.items():
        if delta.dtype not in [torch.float32, torch.float16, torch.bfloat16]:
            compressed[name] = delta
            total_orig += delta.numel() * 4
            total_comp += delta.numel() * 4
            continue

        d = delta.float()
        n = d.numel()

        if n < 10:
            compressed[name] = delta
            total_orig += n * 4
            total_comp += n * 4
            continue

        sensitivity  = layer_sensitivities.get(name, 0)
        is_high_sens = sensitivity > threshold
        ratio        = (config.ADAPTIVE_MAX_RATIO if is_high_sens
                        else config.ADAPTIVE_MIN_RATIO)

        prev_err = error_buf.get(name, None)
        if prev_err is not None:
            d_eff = d + prev_err.to(d.device).view(d.shape)
        else:
            d_eff = d

        ef_norms.append(d_eff.norm().item())

        flat = d_eff.view(-1)
        k    = max(1, int(ratio * n))
        _, top_idx = torch.topk(flat.abs(), k)
        top_val    = flat[top_idx]

        sparse_reconstructed = torch.zeros_like(flat)
        sparse_reconstructed[top_idx] = top_val
        residual = (d_eff.view(-1) - sparse_reconstructed).view(d.shape).detach()
        new_ef[name] = residual.cpu() if config.EF_ON_CPU else residual

        sp = SparsePayload(
            indices=top_idx.cpu().numpy().astype(np.int32),
            values =top_val.cpu().numpy().astype(np.float32),
            shape  =d.shape,
        )
        compressed[name] = sp

        total_orig += n * 4
        total_comp += sp.nbytes
        layer_ratios[name] = {
            'ratio':        ratio,
            'is_high_sens': is_high_sens,
            'sensitivity':  sensitivity,
            'threshold':    threshold,
            'k': k, 'n': n,
        }

    avg_ef_norm = float(np.mean(ef_norms)) if ef_norms else 0.0

    # NEW: Build per-layer sensitivity snapshot for paper figures
    sensitivity_snapshot = {
        'layer_sensitivities': dict(layer_sensitivities),
        'threshold': threshold,
        'layer_ratios': {k: v.copy() for k, v in layer_ratios.items()},
    }

    return compressed, new_ef, total_orig, total_comp, layer_ratios, avg_ef_norm, sensitivity_snapshot


# =============================================================================
# SECTION 8 — SPARSE AGGREGATION
# =============================================================================

def sparse_aggregate(global_model, updates, device):
    base = (global_model.module if isinstance(global_model, nn.DataParallel)
            else global_model)
    state = base.state_dict()
    total_samples = sum(u['n'] for u in updates)

    for key in state:
        if state[key].dtype in [torch.long, torch.int64]:
            continue
        acc = torch.zeros_like(state[key], dtype=torch.float32)
        weight_sum = 0.0
        for u in updates:
            if key not in u['deltas']:
                continue
            w  = u['n'] / total_samples
            sp = u['deltas'][key]
            if isinstance(sp, SparsePayload):
                acc += w * sp.to_dense(device)
            elif torch.is_tensor(sp):
                acc += w * sp.to(device).float()
            weight_sum += w
        if weight_sum > 0:
            state[key] = state[key] + acc.to(state[key].dtype)

    base.load_state_dict(state)


# =============================================================================
# SECTION 9 — LOCAL TRAINING (FedProx)
# =============================================================================

def local_train(client_model, train_loader, device, rnd, global_model_state, mu=0.01):
    initial_state = {k: v.clone() for k, v in client_model.state_dict().items()}
    client_model.train()

    if config.FREEZE_BN_IN_FL:
        client_model.freeze_bn_for_fl()
    if rnd < config.FL_FREEZE_ROUNDS:
        client_model.freeze_backbone()
    else:
        client_model.unfreeze_backbone()

    gw = {k: v.clone().to(device) for k, v in global_model_state.items()
          if v.dtype in [torch.float32, torch.float16]}

    bb_lr   = MODEL_REGISTRY[MODEL_NAME]["backbone_lr"]
    head_lr = MODEL_REGISTRY[MODEL_NAME]["head_lr"]
    optimizer = optim.AdamW(
        client_model.get_param_groups(bb_lr, head_lr),
        weight_decay=config.WEIGHT_DECAY
    )

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([config.POS_WEIGHT], device=device)
    )

    if config.USE_AMP:
        scaler = GradScaler('cuda') if _AMP_NEW else GradScaler()

    for _ in range(config.LOCAL_EPOCHS):
        for x, y in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            if config.USE_AMP:
                ctx = autocast('cuda') if _AMP_NEW else autocast()
                with ctx:
                    loss = criterion(client_model(x), y)
                    prox = sum(
                        torch.norm(p - gw[n_p]) ** 2
                        for n_p, p in client_model.named_parameters()
                        if n_p in gw and p.requires_grad
                    )
                    loss = loss + (mu / 2.0) * prox
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(client_model.parameters(), config.GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss = criterion(client_model(x), y)
                prox = sum(
                    torch.norm(p - gw[n_p]) ** 2
                    for n_p, p in client_model.named_parameters()
                    if n_p in gw and p.requires_grad
                )
                loss = loss + (mu / 2.0) * prox
                loss.backward()
                torch.nn.utils.clip_grad_norm_(client_model.parameters(), config.GRAD_CLIP)
                optimizer.step()

    final_state = client_model.state_dict()
    deltas = {k: final_state[k] - initial_state[k] for k in final_state}
    return deltas, len(train_loader.dataset)


# =============================================================================
# SECTION 10 — FEDERATED TRAINING LOOP (Enhanced with sensitivity tracking)
# =============================================================================

def train_federated(client_datasets, val_loader, test_loader, seed=42):
    set_seed(seed)
    device = config.DEVICE
    t0     = time.time()

    global_model = create_model().to(device)

    print(f"\n  FedAdapt-EF [{MODEL_NAME}] seed={seed}")
    print(f"    FedProx mu={config.FEDPROX_MU} | Adaptive | EF=ON")

    history = {
        'val_auc': [], 'test_auc': [],
        'comm_mb_cumulative': [], 'orig_mb_cumulative': [],
        'compression_ratio': [],
        'ef_norm': [], 'high_sens_pct': [], 'avg_ratio': [],
    }

    # NEW: per-round sensitivity snapshots for paper figures
    sensitivity_history = []

    eval_mask   = []
    total_comm  = 0
    total_orig  = 0
    last_val    = 0.5
    last_test   = 0.5
    ef_buffers  = {cid: {} for cid in client_datasets}

    if config.SAVE_CHECKPOINTS:
        ckpt_dir = MODEL_SAVE_DIR / "checkpoints"
        ckpt_dir.mkdir(parents=True, exist_ok=True)

    eval_rounds   = {0, 5, 10, 15, 20, 24}
    client_batch  = MODEL_REGISTRY[MODEL_NAME]["client_batch"]

    pbar = tqdm(range(config.FL_ROUNDS), desc=f"  FL [{MODEL_NAME}] seed={seed}")

    for rnd in pbar:
        updates            = []
        round_orig         = 0
        round_comp         = 0
        round_ef_norms     = []
        round_high_pcts    = []
        round_avg_ratios   = []
        round_sensitivities = []

        global_state = global_model.state_dict()

        for cid, (train_ds, _) in client_datasets.items():
            train_dl = DataLoader(
                train_ds,
                batch_size=client_batch,
                shuffle=True,
                num_workers=0,
                generator=torch.Generator().manual_seed(seed + rnd + cid),
            )
            client_model = create_client_model(device)
            client_model.load_state_dict(global_state)

            deltas, n = local_train(
                client_model, train_dl, device, rnd,
                global_model_state=global_state, mu=config.FEDPROX_MU,
            )
            compressed, new_ef, orig_b, comp_b, layer_ratios, ef_norm, sens_snap = \
                adaptive_compress_with_ef(deltas, ef_buffers[cid])

            ef_buffers[cid] = new_ef
            round_ef_norms.append(ef_norm)
            round_orig += orig_b
            round_comp += comp_b
            updates.append({'deltas': compressed, 'n': n})

            # Collect sensitivity data from first client only (representative)
            if cid == 0:
                round_sensitivities.append(sens_snap)

            if layer_ratios:
                n_high = sum(1 for v in layer_ratios.values() if v.get('is_high_sens', False))
                round_high_pcts.append(n_high / max(len(layer_ratios), 1) * 100)
                round_avg_ratios.append(
                    np.mean([v['ratio'] for v in layer_ratios.values()])
                )

            del client_model, deltas
            torch.cuda.empty_cache()

        sparse_aggregate(global_model, updates, device)
        total_orig += round_orig
        total_comm += round_comp
        del updates
        torch.cuda.empty_cache()

        # Save sensitivity snapshot for this round
        if round_sensitivities:
            sensitivity_history.append(round_sensitivities[0])

        is_eval = (rnd in eval_rounds)
        if is_eval:
            val_m  = evaluate(global_model, val_loader,  device, return_arrays=False)
            test_m = evaluate(global_model, test_loader, device, return_arrays=False)
            last_val  = val_m['auc']
            last_test = test_m['auc']

        history['val_auc'].append(last_val)
        history['test_auc'].append(last_test)
        history['comm_mb_cumulative'].append(total_comm / 1e6)
        history['orig_mb_cumulative'].append(total_orig / 1e6)
        history['compression_ratio'].append(round_comp / round_orig if round_orig > 0 else 1.0)
        history['ef_norm'].append(float(np.mean(round_ef_norms)))
        history['high_sens_pct'].append(float(np.mean(round_high_pcts)) if round_high_pcts else 0)
        history['avg_ratio'].append(float(np.mean(round_avg_ratios)) if round_avg_ratios else 0.3)
        eval_mask.append(is_eval)

        pbar.set_postfix({
            'val':  f"{last_val:.4f}",
            'test': f"{last_test:.4f}",
            'comm': f"{total_comm/1e6:.0f}MB",
        })

        if config.SAVE_CHECKPOINTS and (rnd + 1) % config.CHECKPOINT_EVERY == 0:
            torch.save({
                'round': rnd + 1, 'model': global_model.state_dict(), 'history': history,
            }, ckpt_dir / f"round{rnd+1}_seed{seed}.pt")

    for key in ['val_auc', 'test_auc']:
        history[key] = interpolate_history(history[key], eval_mask)

    # Add sensitivity history to the main history dict
    history['sensitivity_history'] = sensitivity_history

    final_metrics = evaluate(global_model, test_loader, device, return_arrays=True)
    elapsed = time.time() - t0
    print(f"\n  seed={seed} | AUC={final_metrics['auc']:.4f} "
          f"| F1={final_metrics['f1']:.4f} | Recall={final_metrics['recall']:.4f} "
          f"| Comm={total_comm/1e6:.1f}MB | Time={elapsed/60:.1f}min")

    return {
        'history':       history,
        'final_metrics': final_metrics,
        'total_comm_mb': total_comm / 1e6,
        'total_orig_mb': total_orig / 1e6,
        'elapsed_sec':   elapsed,
        'eval_mask':     eval_mask,
        'model_name':    MODEL_NAME,
        'seed':          seed,
    }


# =============================================================================
# SECTION 11 — AGGREGATE SEEDS + SAVE
# =============================================================================

def aggregate_and_save(seed_results):
    aucs  = [r['final_metrics']['auc']       for r in seed_results]
    f1s   = [r['final_metrics']['f1']        for r in seed_results]
    precs = [r['final_metrics']['precision'] for r in seed_results]
    recs  = [r['final_metrics']['recall']    for r in seed_results]
    accs  = [r['final_metrics']['accuracy']  for r in seed_results]
    comms = [r['total_comm_mb']              for r in seed_results]
    origs = [r['total_orig_mb']              for r in seed_results]

    auc_mean = float(np.mean(aucs))
    auc_std  = float(np.std(aucs))
    reduction = (1 - np.mean(comms) / np.mean(origs)) * 100

    model_summary = {
        'model_name':    MODEL_NAME,
        'description':   MODEL_REGISTRY[MODEL_NAME]['description'],
        'color':         MODEL_REGISTRY[MODEL_NAME]['color'],
        'auc_mean':      auc_mean,
        'auc_std':       auc_std,
        'auc_ci_lo':     auc_mean - 1.96 * auc_std,
        'auc_ci_hi':     auc_mean + 1.96 * auc_std,
        'f1_mean':       float(np.mean(f1s)),
        'prec_mean':     float(np.mean(precs)),
        'rec_mean':      float(np.mean(recs)),
        'acc_mean':      float(np.mean(accs)),
        'comm_mb':       float(np.mean(comms)),
        'orig_mb':       float(np.mean(origs)),
        'reduction_pct': float(reduction),
        'seed_results':  seed_results,
        'elapsed_min':   sum(r['elapsed_sec'] for r in seed_results) / 60,
    }

    save_path = Path(config.SAVE_DIR) / f"results_{MODEL_NAME}.pkl"
    with open(save_path, 'wb') as f:
        pickle.dump(model_summary, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"\n  Saved -> {save_path}")

    print(f"\n{'='*60}")
    print(f"{MODEL_NAME} -- FINAL SUMMARY")
    print(f"  AUC : {auc_mean:.4f} +/- {auc_std:.4f} "
          f"[{auc_mean - 1.96*auc_std:.4f} -- {auc_mean + 1.96*auc_std:.4f}]")
    print(f"  F1  : {np.mean(f1s):.4f} | Recall: {np.mean(recs):.4f} "
          f"| Precision: {np.mean(precs):.4f}")
    print(f"  Comm: {np.mean(comms):.1f} MB | Reduction: {reduction:.1f}%")
    print(f"  Time: {model_summary['elapsed_min']:.1f} min total")
    print(f"{'='*60}")

    return model_summary


# =============================================================================
# SECTION 12 — ALL PAPER FIGURES
# =============================================================================

def generate_all_paper_figures(model_summary):
    """Generate every figure needed for the paper from training results."""
    seed_results = model_summary['seed_results']
    color = model_summary['color']
    rounds = list(range(1, config.FL_ROUNDS + 1))

    print("\n" + "="*60)
    print("GENERATING ALL PAPER FIGURES")
    print("="*60)

    # ─────────────────────────────────────────────────────────────────
    # FIGURE 2: FL Convergence — Test AUC and Val AUC
    # ─────────────────────────────────────────────────────────────────
    print("\n  [Fig 2] FL Convergence...")

    fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5))
    fig2.suptitle('FL Convergence -- DenseNet-121 (FedAdapt-EF)',
                  fontweight='bold', fontsize=12)

    for ax, key, title in zip(axes2, ['test_auc', 'val_auc'],
                               ['(a) Test AUC', '(b) Val AUC']):
        all_curves = [r['history'][key] for r in seed_results]
        for i, curve in enumerate(all_curves):
            ax.plot(rounds, curve, color=color, alpha=0.4, linewidth=1,
                    label=f"seed {seed_results[i]['seed']}")
        mean_curve = np.mean(all_curves, axis=0)
        ax.plot(rounds, mean_curve, color=color, linewidth=2.5, label='Mean')
        ax.axvline(x=3.5, color='red', linestyle=':', alpha=0.4,
                   label='Backbone unfreeze')
        ax.set_xlabel('Round', fontweight='bold')
        ax.set_ylabel('AUC', fontweight='bold')
        ax.set_title(title, fontweight='bold')
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3, linestyle='--')

    plt.tight_layout()
    fig2.savefig(PAPER_DIR / "fig2_fl_convergence.png", dpi=200, bbox_inches='tight')
    fig2.savefig(PAPER_DIR / "fig2_fl_convergence.pdf", dpi=200, bbox_inches='tight')
    plt.close(fig2)
    print("    -> fig2_fl_convergence.png/pdf")

    # ─────────────────────────────────────────────────────────────────
    # FIGURE 3: AUC vs Cumulative Communication
    # ─────────────────────────────────────────────────────────────────
    print("  [Fig 3] AUC vs Cumulative Communication...")

    fig3, ax3 = plt.subplots(figsize=(10, 6))

    for r in seed_results:
        ax3.plot(r['history']['comm_mb_cumulative'],
                 r['history']['test_auc'],
                 color=color, alpha=0.4, linewidth=1.2,
                 label=f"seed {r['seed']}")

    # Mean curve
    mean_comm = np.mean([r['history']['comm_mb_cumulative'] for r in seed_results], axis=0)
    mean_auc  = np.mean([r['history']['test_auc'] for r in seed_results], axis=0)
    ax3.plot(mean_comm, mean_auc, color=color, linewidth=2.5,
             label='FedAdapt-EF (mean)', marker='o', markersize=3)

    # Uncompressed reference
    mean_orig = np.mean([r['history']['orig_mb_cumulative'] for r in seed_results], axis=0)
    ax3.plot(mean_orig, mean_auc, color='gray', linewidth=1.5, linestyle='--',
             label='Uncompressed (same AUC)', alpha=0.5)

    ax3.set_xlabel('Cumulative Communication (MB)', fontweight='bold', fontsize=11)
    ax3.set_ylabel('Test AUC', fontweight='bold', fontsize=11)
    ax3.set_title('Communication Efficiency: AUC vs Cumulative Communication\n'
                  'DenseNet-121, 25 rounds x 5 clients, mean of 2 seeds',
                  fontweight='bold', fontsize=11)
    ax3.legend(fontsize=9)
    ax3.grid(alpha=0.3, linestyle='--')

    plt.tight_layout()
    fig3.savefig(PAPER_DIR / "fig3_auc_vs_communication.png", dpi=200, bbox_inches='tight')
    fig3.savefig(PAPER_DIR / "fig3_auc_vs_communication.pdf", dpi=200, bbox_inches='tight')
    plt.close(fig3)
    print("    -> fig3_auc_vs_communication.png/pdf")

    # ─────────────────────────────────────────────────────────────────
    # FIGURE 5a: EF Buffer Norm
    # ─────────────────────────────────────────────────────────────────
    print("  [Fig 5a] EF Buffer Norm...")

    fig5a, ax5a = plt.subplots(figsize=(8, 5))
    for r in seed_results:
        ax5a.plot(rounds, r['history']['ef_norm'],
                  color=color, alpha=0.5, linewidth=1.5,
                  label=f"seed {r['seed']}")
    mean_ef = np.mean([r['history']['ef_norm'] for r in seed_results], axis=0)
    ax5a.plot(rounds, mean_ef, color=color, linewidth=2.5, label='Mean')
    ax5a.set_xlabel('Round', fontweight='bold')
    ax5a.set_ylabel('EF Buffer Norm', fontweight='bold')
    ax5a.set_title('Error Feedback Buffer Norm -- DenseNet-121\n'
                   'FedAdapt-EF keeps norms near zero (adaptive 50/10 split)',
                   fontweight='bold', fontsize=11)
    ax5a.legend(fontsize=9)
    ax5a.grid(alpha=0.3, linestyle='--')

    plt.tight_layout()
    fig5a.savefig(PAPER_DIR / "fig5a_ef_buffer_norm.png", dpi=200, bbox_inches='tight')
    fig5a.savefig(PAPER_DIR / "fig5a_ef_buffer_norm.pdf", dpi=200, bbox_inches='tight')
    plt.close(fig5a)
    print("    -> fig5a_ef_buffer_norm.png/pdf")

    # ─────────────────────────────────────────────────────────────────
    # FIGURE 5b: Compression Ratio Over Training Rounds
    # ─────────────────────────────────────────────────────────────────
    print("  [Fig 5b] Compression Ratio...")

    fig5b, ax5b = plt.subplots(figsize=(8, 5))
    for r in seed_results:
        ax5b.plot(rounds, r['history']['compression_ratio'],
                  color=color, alpha=0.5, linewidth=1.5,
                  label=f"seed {r['seed']}")
    mean_cr = np.mean([r['history']['compression_ratio'] for r in seed_results], axis=0)
    ax5b.plot(rounds, mean_cr, color=color, linewidth=2.5, label='Mean')
    ax5b.axhline(y=1.0, color='gray', linestyle=':', alpha=0.5, label='No compression')
    ax5b.axvline(x=3.5, color='red', linestyle=':', alpha=0.4, label='Backbone unfreeze')
    ax5b.set_xlabel('Round', fontweight='bold')
    ax5b.set_ylabel('comp_bytes / orig_bytes', fontweight='bold')
    ax5b.set_title('Compression Ratio Over Training Rounds -- DenseNet-121\n'
                   'Self-calibrates from ~0.26 (frozen) to ~0.85 (unfrozen)',
                   fontweight='bold', fontsize=11)
    ax5b.legend(fontsize=9)
    ax5b.grid(alpha=0.3, linestyle='--')

    plt.tight_layout()
    fig5b.savefig(PAPER_DIR / "fig5b_compression_ratio.png", dpi=200, bbox_inches='tight')
    fig5b.savefig(PAPER_DIR / "fig5b_compression_ratio.pdf", dpi=200, bbox_inches='tight')
    plt.close(fig5b)
    print("    -> fig5b_compression_ratio.png/pdf")

    # ─────────────────────────────────────────────────────────────────
    # FIGURE 6: Per-Seed Reproducibility Scatter
    # ─────────────────────────────────────────────────────────────────
    print("  [Fig 6] Per-Seed Reproducibility...")

    fig6, ax6 = plt.subplots(figsize=(6, 5))

    seed_aucs = [r['final_metrics']['auc'] for r in seed_results]
    for j, sa in enumerate(seed_aucs):
        ax6.scatter(0 + (j - 0.5) * 0.15, sa,
                    c=color, s=120, edgecolors='black', linewidth=1, zorder=5,
                    marker='o', label=f"seed {seed_results[j]['seed']}")
    ax6.scatter(0, model_summary['auc_mean'], c=color, s=200, marker='D',
                edgecolors='black', linewidth=1.5, zorder=6, label='Mean')
    if len(seed_aucs) > 1:
        ax6.plot([0, 0], [min(seed_aucs), max(seed_aucs)],
                 color=color, linewidth=2, alpha=0.5)
    ax6.set_xlim([-0.5, 0.5])
    ax6.set_xticks([0])
    ax6.set_xticklabels(['FedAdapt-EF\n(DenseNet-121)'], fontsize=10)
    ax6.set_ylabel('Test AUC', fontweight='bold')
    ax6.set_title(f'Per-Seed Reproducibility -- DenseNet-121\n'
                  f'AUC = {model_summary["auc_mean"]:.4f} +/- {model_summary["auc_std"]:.4f}',
                  fontweight='bold', fontsize=11)
    ax6.legend(fontsize=9)
    ax6.grid(axis='y', alpha=0.3, linestyle='--')

    plt.tight_layout()
    fig6.savefig(PAPER_DIR / "fig6_per_seed_reproducibility.png", dpi=200, bbox_inches='tight')
    plt.close(fig6)
    print("    -> fig6_per_seed_reproducibility.png")

    # ─────────────────────────────────────────────────────────────────
    # NEW: Layer-wise Sensitivity Heatmap
    # ─────────────────────────────────────────────────────────────────
    print("  [NEW] Layer-wise Sensitivity Heatmap...")

    # Get sensitivity data from the LAST round of the first seed
    sens_hist = seed_results[0]['history'].get('sensitivity_history', [])
    if sens_hist and len(sens_hist) > 0:
        last_sens = sens_hist[-1]  # Last round
        layer_sens = last_sens.get('layer_sensitivities', {})
        threshold = last_sens.get('threshold', 0)

        if layer_sens:
            # Group layers by DenseNet-121 architecture
            group_map = OrderedDict()
            group_sens = OrderedDict()

            for lname, sval in sorted(layer_sens.items()):
                # Classify layer into architectural group
                if 'features.conv0' in lname or 'features.norm0' in lname:
                    group = 'Conv0 (Initial)'
                elif 'denseblock1' in lname:
                    group = 'DenseBlock1'
                elif 'transition1' in lname:
                    group = 'Transition1'
                elif 'denseblock2' in lname:
                    group = 'DenseBlock2'
                elif 'transition2' in lname:
                    group = 'Transition2'
                elif 'denseblock3' in lname:
                    group = 'DenseBlock3'
                elif 'transition3' in lname:
                    group = 'Transition3'
                elif 'denseblock4' in lname:
                    group = 'DenseBlock4'
                elif 'features.norm5' in lname or 'norm' in lname.lower():
                    group = 'BatchNorm (final)'
                elif 'classifier' in lname:
                    group = 'Classifier'
                else:
                    group = 'Other'

                if group not in group_sens:
                    group_sens[group] = []
                group_sens[group].append(sval)

            # Average sensitivity per group
            group_names = list(group_sens.keys())
            group_means = [float(np.mean(v)) for v in group_sens.values()]
            group_counts = [len(v) for v in group_sens.values()]

            fig_sens, (ax_s1, ax_s2) = plt.subplots(1, 2, figsize=(16, 6),
                                                     gridspec_kw={'width_ratios': [2, 1]})
            fig_sens.suptitle('Layer-wise Frobenius Sensitivity Profile -- DenseNet-121\n'
                              's_l = ||delta_w_l||_F / sqrt(n_l)   (Eq. 2 in paper)',
                              fontweight='bold', fontsize=12)

            # Color by type
            type_colors_map = {
                'Conv0 (Initial)': '#E74C3C',
                'DenseBlock1': '#3498DB', 'DenseBlock2': '#3498DB',
                'DenseBlock3': '#3498DB', 'DenseBlock4': '#3498DB',
                'Transition1': '#F39C12', 'Transition2': '#F39C12',
                'Transition3': '#F39C12',
                'BatchNorm (final)': '#95A5A6',
                'Classifier': '#9B59B6',
                'Other': '#888888',
            }
            bar_colors = [type_colors_map.get(g, '#888') for g in group_names]
            keep_labels = ['50% (High)' if s > threshold else '10% (Low)' for s in group_means]

            bars = ax_s1.barh(range(len(group_names)), group_means, color=bar_colors,
                              edgecolor='black', linewidth=0.8, alpha=0.88)
            ax_s1.axvline(x=threshold, color='red', linestyle='--', linewidth=2,
                          label=f'Median threshold = {threshold:.4f}')
            ax_s1.set_yticks(range(len(group_names)))
            ax_s1.set_yticklabels([f"{g} ({c} layers)" for g, c in zip(group_names, group_counts)],
                                  fontsize=9)
            ax_s1.set_xlabel('Normalised Frobenius Sensitivity Score', fontweight='bold')
            ax_s1.set_title('(a) Per-Group Sensitivity (Round 25)', fontweight='bold')
            ax_s1.invert_yaxis()
            ax_s1.grid(axis='x', alpha=0.3, linestyle='--')

            for i, (bar, kr, s) in enumerate(zip(bars, keep_labels, group_means)):
                kr_color = '#1a7a1a' if '50%' in kr else '#cc3333'
                ax_s1.text(s + max(group_means)*0.02, i, kr, va='center',
                           fontsize=8, color=kr_color, fontweight='bold')

            # Legend patches
            patches = [
                mpatches.Patch(color='#E74C3C', label='Conv/Initial'),
                mpatches.Patch(color='#3498DB', label='Dense Block'),
                mpatches.Patch(color='#F39C12', label='Transition'),
                mpatches.Patch(color='#95A5A6', label='BatchNorm'),
                mpatches.Patch(color='#9B59B6', label='Classifier'),
                plt.Line2D([0],[0], color='red', linestyle='--', linewidth=2,
                           label=f'Median = {threshold:.4f}'),
            ]
            ax_s1.legend(handles=patches, fontsize=7.5, loc='lower right', ncol=2)

            # Heatmap
            sens_arr = np.array(group_means).reshape(-1, 1)
            cmap = LinearSegmentedColormap.from_list('sens', ['#2ECC71', '#F39C12', '#E74C3C'])
            vmax = max(group_means) * 1.1 if group_means else 1.0
            im = ax_s2.imshow(sens_arr, cmap=cmap, aspect='auto', vmin=0, vmax=vmax)
            ax_s2.set_yticks(range(len(group_names)))
            ax_s2.set_yticklabels(group_names, fontsize=9)
            ax_s2.set_xticks([0])
            ax_s2.set_xticklabels(['s_l'], fontsize=10)
            ax_s2.set_title('(b) Heatmap', fontweight='bold')
            plt.colorbar(im, ax=ax_s2, shrink=0.8, label='Sensitivity')
            for i, s in enumerate(group_means):
                ax_s2.text(0, i, f'{s:.4f}', ha='center', va='center',
                           fontsize=8, fontweight='bold',
                           color='white' if s > vmax*0.5 else 'black')

            plt.tight_layout()
            fig_sens.savefig(PAPER_DIR / "layer_sensitivity_heatmap.png", dpi=200, bbox_inches='tight')
            fig_sens.savefig(PAPER_DIR / "layer_sensitivity_heatmap.pdf", dpi=200, bbox_inches='tight')
            plt.close(fig_sens)
            print("    -> layer_sensitivity_heatmap.png/pdf")

            # Print medical insight
            print(f"\n  SENSITIVITY ANALYSIS (Round 25):")
            print(f"    Median threshold: {threshold:.6f}")
            for g, s, kr in zip(group_names, group_means, keep_labels):
                print(f"    {g:30s}  s={s:.6f}  {kr}")
        else:
            print("    No layer sensitivity data found in history")
    else:
        print("    No sensitivity history recorded")

    # ─────────────────────────────────────────────────────────────────
    # NEW: Clinical Confusion Matrix
    # ─────────────────────────────────────────────────────────────────
    print("\n  [NEW] Clinical Confusion Matrix...")

    # Use the first seed's final_metrics (has y_true, y_pred)
    fm = seed_results[0]['final_metrics']
    if 'cm' in fm:
        cm = fm['cm']
    elif 'labels' in fm and 'preds' in fm:
        cm = confusion_matrix(fm['labels'], fm['preds'])
    else:
        cm = None

    if cm is not None:
        tn, fp, fn, tp = cm.ravel()
        total = tn + fp + fn + tp
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
        npv = tn / (tn + fn) if (tn + fn) > 0 else 0
        fnr = fn / (tp + fn) if (tp + fn) > 0 else 0

        fig_cm, axes_cm = plt.subplots(1, 2, figsize=(14, 5.5))
        fig_cm.suptitle('Clinical Error Analysis -- FedAdapt-EF (DenseNet-121, Seed 42)',
                        fontweight='bold', fontsize=12)

        # Matrix
        ax_cm = axes_cm[0]
        im = ax_cm.imshow(cm, cmap='Blues', interpolation='nearest')
        ax_cm.set_xticks([0, 1])
        ax_cm.set_yticks([0, 1])
        ax_cm.set_xticklabels(['Predicted\nNo DR', 'Predicted\nDR+'], fontsize=10)
        ax_cm.set_yticklabels(['Actual\nNo DR', 'Actual\nDR+'], fontsize=10)
        ax_cm.set_title('(a) Confusion Matrix', fontweight='bold')

        cell_labels = [['TN', 'FP'], ['FN', 'TP']]
        for i in range(2):
            for j in range(2):
                clr = 'white' if cm[i, j] > cm.max()/2 else 'black'
                ax_cm.text(j, i, f'{cell_labels[i][j]}\n{cm[i,j]:,}',
                           ha='center', va='center', fontsize=12,
                           fontweight='bold', color=clr)
        plt.colorbar(im, ax=ax_cm, shrink=0.8)

        # Text panel
        ax_t = axes_cm[1]
        ax_t.axis('off')
        lines = [
            "CLINICAL ERROR ANALYSIS",
            "-" * 48,
            f"Test set: {total:,} samples",
            f"  No DR: {tn+fp:,}  |  DR+: {tp+fn:,}",
            "",
            f"True Positives  (DR+ correct):     {tp:,}",
            f"True Negatives  (No DR correct):   {tn:,}",
            f"False Positives (unnecessary ref):  {fp:,}",
            f"False Negatives (MISSED DR):        {fn:,}",
            "",
            "-" * 48,
            f"Sensitivity (Recall):  {sensitivity:.4f}",
            f"Specificity:           {specificity:.4f}",
            f"PPV (Precision):       {ppv:.4f}",
            f"NPV:                   {npv:.4f}",
            f"False Negative Rate:   {fnr:.4f}  ({fn} missed)",
            "",
            "-" * 48,
            "CLINICAL INTERPRETATION:",
            f"  {fn} DR+ patients would be missed",
            f"  {fp} healthy patients unnecessarily referred",
            f"  Miss rate ({fnr:.1%}) acceptable for screening",
            f"  Precision ({ppv:.1%}) prevents overload",
        ]
        ax_t.text(0.02, 0.97, "\n".join(lines), fontsize=8.5,
                  family="monospace", va="top", transform=ax_t.transAxes,
                  bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.9))

        plt.tight_layout()
        fig_cm.savefig(PAPER_DIR / "clinical_confusion_matrix.png", dpi=200, bbox_inches='tight')
        fig_cm.savefig(PAPER_DIR / "clinical_confusion_matrix.pdf", dpi=200, bbox_inches='tight')
        plt.close(fig_cm)
        print("    -> clinical_confusion_matrix.png/pdf")
    else:
        print("    No confusion matrix data available")

    # ─────────────────────────────────────────────────────────────────
    # NEW: ROC Curve
    # ─────────────────────────────────────────────────────────────────
    print("  [NEW] ROC Curve...")

    if 'fpr' in fm and 'tpr' in fm:
        fig_roc, ax_roc = plt.subplots(figsize=(7, 6))
        ax_roc.plot(fm['fpr'], fm['tpr'], color=color, linewidth=2.5,
                    label=f'FedAdapt-EF (AUC = {fm["auc"]:.4f})')
        ax_roc.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
        ax_roc.set_xlabel('False Positive Rate', fontweight='bold')
        ax_roc.set_ylabel('True Positive Rate', fontweight='bold')
        ax_roc.set_title('ROC Curve -- FedAdapt-EF (DenseNet-121, Seed 42)',
                         fontweight='bold')
        ax_roc.legend(fontsize=10)
        ax_roc.grid(alpha=0.3)
        plt.tight_layout()
        fig_roc.savefig(PAPER_DIR / "roc_curve.png", dpi=200, bbox_inches='tight')
        fig_roc.savefig(PAPER_DIR / "roc_curve.pdf", dpi=200, bbox_inches='tight')
        plt.close(fig_roc)
        print("    -> roc_curve.png/pdf")

    # ─────────────────────────────────────────────────────────────────
    # NEW: Communication Breakdown (cumulative comm vs uncompressed)
    # ─────────────────────────────────────────────────────────────────
    print("  [NEW] Communication Breakdown...")

    fig_cb, ax_cb = plt.subplots(figsize=(10, 5))
    for r in seed_results:
        ax_cb.plot(rounds, r['history']['comm_mb_cumulative'],
                   color=color, alpha=0.5, linewidth=1.5,
                   label=f"Compressed (seed {r['seed']})")
        ax_cb.plot(rounds, r['history']['orig_mb_cumulative'],
                   color='gray', alpha=0.4, linewidth=1.5, linestyle='--',
                   label=f"Uncompressed (seed {r['seed']})" if r == seed_results[0] else None)

    mean_c = np.mean([r['history']['comm_mb_cumulative'] for r in seed_results], axis=0)
    mean_o = np.mean([r['history']['orig_mb_cumulative'] for r in seed_results], axis=0)
    ax_cb.fill_between(rounds, mean_c, mean_o, alpha=0.1, color=color,
                       label=f'Savings ({model_summary["reduction_pct"]:.1f}%)')
    ax_cb.set_xlabel('Round', fontweight='bold')
    ax_cb.set_ylabel('Cumulative Communication (MB)', fontweight='bold')
    ax_cb.set_title(f'Communication Savings -- DenseNet-121 FedAdapt-EF\n'
                    f'Total: {model_summary["comm_mb"]:.0f} MB compressed vs '
                    f'{model_summary["orig_mb"]:.0f} MB uncompressed '
                    f'(reduction {model_summary["reduction_pct"]:.1f}%)',
                    fontweight='bold', fontsize=10)
    ax_cb.legend(fontsize=8)
    ax_cb.grid(alpha=0.3, linestyle='--')
    plt.tight_layout()
    fig_cb.savefig(PAPER_DIR / "communication_breakdown.png", dpi=200, bbox_inches='tight')
    plt.close(fig_cb)
    print("    -> communication_breakdown.png")

    # ─────────────────────────────────────────────────────────────────
    # NEW: High Sensitivity % and Avg Keep Ratio over rounds
    # ─────────────────────────────────────────────────────────────────
    print("  [NEW] Adaptive Compression Diagnostics...")

    fig_ad, (ax_ad1, ax_ad2) = plt.subplots(1, 2, figsize=(14, 5))
    fig_ad.suptitle('Adaptive Compression Diagnostics -- DenseNet-121',
                    fontweight='bold', fontsize=12)

    for r in seed_results:
        ax_ad1.plot(rounds, r['history']['high_sens_pct'],
                    color=color, alpha=0.5, linewidth=1.5)
    mean_hp = np.mean([r['history']['high_sens_pct'] for r in seed_results], axis=0)
    ax_ad1.plot(rounds, mean_hp, color=color, linewidth=2.5, label='Mean')
    ax_ad1.axhline(y=50, color='gray', linestyle=':', alpha=0.5, label='50% line')
    ax_ad1.set_xlabel('Round', fontweight='bold')
    ax_ad1.set_ylabel('% Layers Classified High-Sensitivity', fontweight='bold')
    ax_ad1.set_title('(a) High-Sensitivity Layer Fraction', fontweight='bold')
    ax_ad1.legend(fontsize=9)
    ax_ad1.grid(alpha=0.3, linestyle='--')

    for r in seed_results:
        ax_ad2.plot(rounds, r['history']['avg_ratio'],
                    color=color, alpha=0.5, linewidth=1.5)
    mean_ar = np.mean([r['history']['avg_ratio'] for r in seed_results], axis=0)
    ax_ad2.plot(rounds, mean_ar, color=color, linewidth=2.5, label='Mean')
    ax_ad2.axhline(y=0.5, color='red', linestyle=':', alpha=0.3, label='r_max=0.5')
    ax_ad2.axhline(y=0.1, color='blue', linestyle=':', alpha=0.3, label='r_min=0.1')
    ax_ad2.set_xlabel('Round', fontweight='bold')
    ax_ad2.set_ylabel('Average Keep Ratio', fontweight='bold')
    ax_ad2.set_title('(b) Mean Keep Ratio Across Layers', fontweight='bold')
    ax_ad2.legend(fontsize=9)
    ax_ad2.grid(alpha=0.3, linestyle='--')

    plt.tight_layout()
    fig_ad.savefig(PAPER_DIR / "adaptive_compression_diagnostics.png", dpi=200, bbox_inches='tight')
    plt.close(fig_ad)
    print("    -> adaptive_compression_diagnostics.png")

    # ─────────────────────────────────────────────────────────────────
    # COMBINED 3-panel convergence (for quick overview)
    # ─────────────────────────────────────────────────────────────────
    print("  [Combined] 3-panel convergence plot...")

    fig_3p, axes_3p = plt.subplots(1, 3, figsize=(16, 4.5))
    fig_3p.suptitle(f'FedAdapt-EF -- DenseNet-121\n{model_summary["description"]}',
                    fontsize=12, fontweight='bold')

    # AUC convergence
    ax = axes_3p[0]
    all_test = [r['history']['test_auc'] for r in seed_results]
    for i, curve in enumerate(all_test):
        ax.plot(rounds, curve, color=color, alpha=0.4, linewidth=1,
                label=f"seed {seed_results[i]['seed']}")
    ax.plot(rounds, np.mean(all_test, axis=0), color=color, linewidth=2.5, label='Mean')
    ax.set_title('(a) Test AUC Convergence')
    ax.set_xlabel('Round'); ax.set_ylabel('AUC')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    # Cumulative communication
    ax = axes_3p[1]
    for r in seed_results:
        ax.plot(rounds, r['history']['comm_mb_cumulative'],
                color=color, alpha=0.5, linewidth=1)
        ax.plot(rounds, r['history']['orig_mb_cumulative'],
                color='gray', alpha=0.3, linewidth=1, linestyle='--')
    ax.set_title('(b) Cumulative Communication (MB)')
    ax.set_xlabel('Round'); ax.set_ylabel('MB')
    ax.grid(alpha=0.3)

    # EF buffer norm
    ax = axes_3p[2]
    for r in seed_results:
        ax.plot(rounds, r['history']['ef_norm'],
                color=color, alpha=0.6, linewidth=1.5)
    ax.set_title('(c) EF Buffer Norm')
    ax.set_xlabel('Round'); ax.set_ylabel('Norm')
    ax.grid(alpha=0.3)

    plt.tight_layout()
    fig_3p.savefig(PAPER_DIR / "combined_convergence.png", dpi=150, bbox_inches='tight')
    plt.close(fig_3p)
    print("    -> combined_convergence.png")

    # ─────────────────────────────────────────────────────────────────
    # HYPERPARAMETER TABLE (Table 4)
    # ─────────────────────────────────────────────────────────────────
    print("  [Table 4] Hyperparameter Table...")

    hp_data = [
        ["Backbone",                "DenseNet-121 (ImageNet-pretrained)"],
        ["Head",                    "1024 -> 512 -> 1, ReLU, Dropout 0.4"],
        ["Input",                   "224 x 224 x 3"],
        ["Clients K / Rounds T / Epochs E", "5 / 25 / 1"],
        ["Non-IID alpha (Dirichlet)", "0.5"],
        ["Optimiser",               "AdamW (lambda=1e-4)"],
        ["Backbone LR / Head LR",   "3e-5 / 3e-4"],
        ["Batch size / Grad clip",  "24 / 1.0"],
        ["w+ / FedProx mu",        "2.0 / 0.01"],
        ["r_max / r_min / fixed TopK", "0.5 / 0.1 / 0.3"],
        ["Sensitivity threshold",  "per-round median"],
        ["EF buffer storage",      "CPU RAM"],
        ["Seeds / Backbone freeze", "{42, 123} / rounds < 3"],
    ]

    fig_hp, ax_hp = plt.subplots(figsize=(10, len(hp_data)*0.45 + 1.5))
    ax_hp.axis('off')
    tbl = ax_hp.table(cellText=hp_data, colLabels=['Parameter', 'Value'],
                      cellLoc='left', loc='center', bbox=[0, 0, 1, 1])
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    for j in range(2):
        tbl[0, j].set_facecolor('#1F497D')
        tbl[0, j].set_text_props(color='white', fontweight='bold')
    for i in range(len(hp_data)):
        fill = '#F9FBFF' if i % 2 == 0 else '#FFFFFF'
        for j in range(2):
            tbl[i+1, j].set_facecolor(fill)
    ax_hp.set_title('Table 4: Hyperparameters', fontsize=11, fontweight='bold', pad=15)
    plt.tight_layout()
    fig_hp.savefig(PAPER_DIR / "table4_hyperparameters.png", dpi=200, bbox_inches='tight')
    plt.close(fig_hp)
    print("    -> table4_hyperparameters.png")

    # ─────────────────────────────────────────────────────────────────
    # SUMMARY CSV
    # ─────────────────────────────────────────────────────────────────
    summary_row = {
        'model': MODEL_NAME,
        'auc_mean': model_summary['auc_mean'],
        'auc_std': model_summary['auc_std'],
        'f1_mean': model_summary['f1_mean'],
        'recall_mean': model_summary['rec_mean'],
        'precision_mean': model_summary['prec_mean'],
        'accuracy_mean': model_summary['acc_mean'],
        'comm_mb': model_summary['comm_mb'],
        'orig_mb': model_summary['orig_mb'],
        'reduction_pct': model_summary['reduction_pct'],
        'elapsed_min': model_summary['elapsed_min'],
    }
    pd.DataFrame([summary_row]).to_csv(PAPER_DIR / "densenet121_summary.csv", index=False)
    print("    -> densenet121_summary.csv")

    print(f"\n  All paper figures saved to: {PAPER_DIR}/")
    print(f"  Files generated:")
    for f in sorted(PAPER_DIR.glob("*")):
        print(f"    {f.name}  ({f.stat().st_size // 1024} KB)")


# =============================================================================
# SECTION 13 — ENTRY POINT
# =============================================================================

if __name__ == "__main__":
    print("\n" + "="*60)
    print(f"NB9-ENHANCED: FedAdapt-EF DenseNet-121")
    print(f"  {MODEL_REGISTRY[MODEL_NAME]['description']}")
    print("="*60)

    # Load data
    print("\nLoading dataset...")
    train_df, val_df, test_df = load_and_split_data()

    val_ds  = DRDataset(val_df,  get_transforms('val'))
    test_ds = DRDataset(test_df, get_transforms('val'))
    val_loader  = DataLoader(val_ds,  batch_size=config.BATCH_SIZE, shuffle=False,
                             num_workers=config.NUM_WORKERS, pin_memory=config.PIN_MEMORY)
    test_loader = DataLoader(test_ds, batch_size=config.BATCH_SIZE, shuffle=False,
                             num_workers=config.NUM_WORKERS, pin_memory=config.PIN_MEMORY)

    # Run both seeds
    seed_results = []
    for seed in config.SEEDS:
        print(f"\n{'='*40}")
        print(f"  Seed {seed}")
        print(f"{'='*40}")
        client_datasets = make_non_iid_splits(
            train_df, config.NUM_CLIENTS, config.NON_IID_ALPHA, seed=seed
        )
        result = train_federated(
            client_datasets, val_loader, test_loader, seed=seed
        )
        seed_results.append(result)
        gc.collect()
        torch.cuda.empty_cache()

    # Aggregate and save
    model_summary = aggregate_and_save(seed_results)

    # Generate ALL paper figures
    generate_all_paper_figures(model_summary)

    # Final summary
    print(f"""
{'='*70}
DONE -- NB9-ENHANCED COMPLETE
{'='*70}

Results pkl: {Path(config.SAVE_DIR) / f'results_{MODEL_NAME}.pkl'}

Paper figures: {PAPER_DIR}/
  fig2_fl_convergence.png/pdf        -- FL convergence (Fig. 2)
  fig3_auc_vs_communication.png/pdf  -- AUC vs comm (Fig. 3)
  fig5a_ef_buffer_norm.png/pdf       -- EF buffer norm (Fig. 5a)
  fig5b_compression_ratio.png/pdf    -- Compression ratio (Fig. 5b)
  fig6_per_seed_reproducibility.png  -- Per-seed scatter (Fig. 6)
  layer_sensitivity_heatmap.png/pdf  -- NEW: Layer sensitivity
  clinical_confusion_matrix.png/pdf  -- NEW: Confusion matrix
  roc_curve.png/pdf                  -- NEW: ROC curve
  communication_breakdown.png        -- NEW: Comm savings area
  adaptive_compression_diagnostics.png -- NEW: High-sens % + avg ratio
  combined_convergence.png           -- Quick overview
  table4_hyperparameters.png         -- Table 4

RESULTS:
  AUC:       {model_summary['auc_mean']:.4f} +/- {model_summary['auc_std']:.4f}
  F1:        {model_summary['f1_mean']:.4f}
  Recall:    {model_summary['rec_mean']:.4f}
  Precision: {model_summary['prec_mean']:.4f}
  Comm:      {model_summary['comm_mb']:.0f} MB (reduction {model_summary['reduction_pct']:.1f}%)
  Time:      {model_summary['elapsed_min']:.1f} min
""")

timm 1.0.25 already installed
NB9-ENHANCED: FedAdapt-EF DenseNet-121 + Paper Diagnostics
PyTorch: 2.10.0+cu128 | timm: 1.0.25
CUDA: True
  GPU 0: Tesla T4
  GPU 1: Tesla T4

Model: densenet121
  DenseNet-121 (8M) -- Medical imaging standard
  client_batch=24

Config ready
  Seeds:      [2026, 3407]
  FL rounds:  25
  Save dir:   /kaggle/working/results/multimodel/densenet121
  Paper figs: /kaggle/working/results/paper_figures

NB9-ENHANCED: FedAdapt-EF DenseNet-121
  DenseNet-121 (8M) -- Medical imaging standard

Loading dataset...

Dataset: 16652 images | DR+: 8422 (50.6%)
  Broken paths (12/20 exist) -- scanning /kaggle/input/ ...
    [competitions]: 5,590 images
    [datasets]: 14,094 images
  Total indexed: 19,684 images
  After remap: 20/20 valid
  Train: 11656 | Val: 2498 | Test: 2498

  Seed 2026


model.safetensors:   0%|          | 0.00/32.3M [00:00<?, ?B/s]

  densenet121: feature_dim=1024
  Created densenet121 | trainable params: 7,479,169

  FedAdapt-EF [densenet121] seed=2026
    FedProx mu=0.01 | Adaptive | EF=ON


  FL [densenet121] seed=2026:   0%|          | 0/25 [00:00<?, ?it/s]

  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densen

  FL [densenet121] seed=3407:   0%|          | 0/25 [00:00<?, ?it/s]

  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densenet121: feature_dim=1024
  densen